# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima8211/ML_Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distributions

I reviewed the distributions of the main fields that may be useful for signal testing: previous 30-day impressions, content age, days since last update, CTR, and average position.

The fields are not evenly distributed. In particular, traffic/exposure and content-age related fields can have long or heavy tails, so simple thresholds should be treated as practical review rules rather than universal cutoffs.

The distribution check is used to understand the data before testing signals.


In [4]:
!git clone https://github.com/Fatima8211/ML_Internship.git

Cloning into 'ML_Internship'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 161 (delta 65), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (161/161), 1.90 MiB | 7.84 MiB/s, done.
Resolving deltas: 100% (65/65), done.


In [1]:
import os

for root, dirs, files in os.walk("/content/ML_Internship"):
    for file in files:
        if file == "content_refresh_anonymized.csv":
            print("Dataset found:")
            print(os.path.join(root, file))

In [5]:
import pandas as pd
import numpy as np

csv_path = "/content/ML_Internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

Dataset loaded successfully!
Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [6]:
import pandas as pd
import numpy as np

csv_path = "/content/ML_Internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

distribution_cols = [
    "impressions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

missing_cols = [
    col for col in distribution_cols
    if col not in df.columns
]

if missing_cols:
    raise KeyError(f"Missing columns: {missing_cols}")

distribution_summary = (
    df[distribution_cols]
    .describe()
    .T
)

distribution_summary["missing"] = df[distribution_cols].isna().sum()
distribution_summary["missing_pct"] = (
    distribution_summary["missing"] / len(df) * 100
).round(2)

display(distribution_summary)

Dataset loaded successfully!
Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


,count,mean,std,min,25%,50%,75%,max,missing,missing_pct
impressions_prev_30d,30000.0,1783.078500,6150.429511,0.0,19.0,210.00,1143.00,218786.0,0,0.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0,0,0.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.0,0,0.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0,0,0.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0,0,0.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*
### The flag-linked test

I used staleness as the flag-linked signal because refresh-oriented review logic can use how long it has been since a content page was last updated.

The test compares pages that have not been updated for at least 180 days with pages updated more recently. The declining proxy is used only to test whether the observed data supports the assumption behind the signal.

This is directional evidence only. A higher declining rate does not prove that staleness causes performance decline.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# ML-06 — Section 3: The flag-linked test

import numpy as np
import pandas as pd

# Create the evaluation proxy locally.
# IMPORTANT: This is used only to test the signal,
# not as a feature in the baseline score.
flag_linked = df.copy()

flag_linked["declining_proxy"] = (
    flag_linked["impressions_last_30d"]
    < 0.8 * flag_linked["impressions_prev_30d"]
).astype(int)

# Flag-linked signal: staleness
flag_linked["staleness_group"] = np.where(
    flag_linked["days_since_last_update"] >= 180,
    "180_plus_days",
    "under_180_days"
)

# Compare declining rates
flag_linked_result = (
    flag_linked
    .groupby("staleness_group", observed=True)["declining_proxy"]
    .agg(
        n="size",
        declining_rate="mean"
    )
    .reset_index()
)

flag_linked_result["declining_rate_pct"] = (
    flag_linked_result["declining_rate"] * 100
).round(2)

display(flag_linked_result)

# Calculate directional difference
rates = flag_linked_result.set_index("staleness_group")[
    "declining_rate"
]

stale_rate = rates.get("180_plus_days", np.nan)
recent_rate = rates.get("under_180_days", np.nan)

if pd.notna(stale_rate) and pd.notna(recent_rate):

    gap = (stale_rate - recent_rate) * 100

    print(
        f"Staleness declining-rate gap: "
        f"{gap:.2f} percentage points"
    )

    if gap > 0:
        print("Verdict: CONFIRMED")
        print(
            "The data directionally supports the assumption "
            "that stale pages are more likely to be declining."
        )
    elif gap < 0:
        print("Verdict: OPPOSITE")
        print(
            "The data directionally goes against the assumption "
            "that stale pages are more likely to be declining."
        )
    else:
        print("Verdict: MIXED")
        print(
            "The two groups have the same observed declining rate."
        )

else:
    print("Verdict: FALSE")
    print("The required groups could not be evaluated.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,staleness_group,n,declining_rate,declining_rate_pct
0,180_plus_days,174,0.471264,47.13
1,under_180_days,29826,0.542480,54.25


Staleness declining-rate gap: -7.12 percentage points
Verdict: OPPOSITE
The data directionally goes against the assumption that stale pages are more likely to be declining.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### The flag-linked test

I tested **staleness** as the flag-linked signal. The rule uses `days_since_last_update >= 180` to identify pages that have not been updated recently.

The test compares the declining rate of stale pages with pages updated within the last 180 days. The result is treated as directional evidence only: it can support or weaken the assumption behind the signal, but it does not establish causation.

The declining proxy is used only for this evaluation and is not used as an input feature for the baseline ranking score.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# ML-06 — Section 3: The flag-linked test

import numpy as np
import pandas as pd

# Create evaluation proxy only
# It is NOT used as a feature in the signal or baseline score.
flag_linked = df.copy()

flag_linked["declining_proxy"] = (
    flag_linked["impressions_last_30d"]
    < 0.8 * flag_linked["impressions_prev_30d"]
).astype(int)

# Flag-linked signal: staleness
flag_linked["staleness_group"] = np.where(
    flag_linked["days_since_last_update"] >= 180,
    "180_plus_days",
    "under_180_days"
)

# Compare the two groups
flag_linked_result = (
    flag_linked
    .groupby("staleness_group", observed=True)["declining_proxy"]
    .agg(
        n="size",
        declining_rate="mean"
    )
    .reset_index()
)

# Convert rate to percentage
flag_linked_result["declining_rate_pct"] = (
    flag_linked_result["declining_rate"] * 100
).round(2)

print("Flag-linked test — Staleness")
display(flag_linked_result)

# Calculate difference between the groups
rates = flag_linked_result.set_index("staleness_group")[
    "declining_rate"
]

stale_rate = rates.get("180_plus_days", np.nan)
recent_rate = rates.get("under_180_days", np.nan)

if pd.notna(stale_rate) and pd.notna(recent_rate):

    gap = (stale_rate - recent_rate) * 100

    print(
        f"Staleness declining-rate gap: "
        f"{gap:.2f} percentage points"
    )

    if gap > 0:
        print("Verdict: CONFIRMED")
    elif gap < 0:
        print("Verdict: OPPOSITE")
    else:
        print("Verdict: MIXED")

else:
    print("Verdict: FALSE")
    print("Required comparison groups could not be evaluated.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Flag-linked test — Staleness


,staleness_group,n,declining_rate,declining_rate_pct
0,180_plus_days,174,0.471264,47.13
1,under_180_days,29826,0.542480,54.25


Staleness declining-rate gap: -7.12 percentage points
Verdict: OPPOSITE


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### What this means in practice

The signal audit provides directional evidence about which observable signals may be useful when prioritizing content for review. Staleness, CTR/position, and previous-period visibility can help identify pages that deserve attention, but these signals should be treated as decision-support indicators rather than proof that a page needs a specific intervention.

The content team should use the signals to decide which pages to review first and then validate the reason manually. The audit does not establish causation, and temporary changes, search intent, seasonality, or other factors may make an individual signal misleading.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# ML-06 — Section 4: Practical summary

print("Practical takeaway:")
print(
    "Use the audited signals as prioritization aids, "
    "then validate the page manually before taking action."
)
print(
    "The observed relationships are directional and "
    "should not be interpreted as causal."
)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Practical takeaway:
Use the audited signals as prioritization aids, then validate the page manually before taking action.
The observed relationships are directional and should not be interpreted as causal.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.